# T-1 Pre-Burst Analysis

Entrar al cierre de T-1 antes del burst. Señales: categorías Finviz + barra diaria (compresión, vol, distancia de máximos).

**Pregunta clave**: ¿hay edge en comprar el día antes?
**Grupo de control**: todos los ticker-days Finviz que NO burstaron al día siguiente.

## 1. Config

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

TZ = "America/New_York"
DB_DAILY  = Path("/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_daily_cache.db")
DB_RTH    = Path("/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_intraday_cache.db")
FINVIZ_DB = Path.home() / "Library/Application Support/finviz-dashboard/finviz_snapshots.db"

SIGNAL_CATS   = ["Unusual Volume","New High","Most Volatile","Most Active","Overbought","Insider Buying","Upgrades"]
NEGATIVE_CATS = ["Top Losers","New Low","Oversold","Downgrades","Insider Selling"]
print("Config OK")

## 2. Cargar datos

In [ ]:
with sqlite3.connect(DB_DAILY) as conn:
    daily = pd.read_sql("SELECT ticker, bar_date, open, high, low, close, volume FROM daily_bars ORDER BY ticker, bar_date", conn)
daily = daily.sort_values(["ticker","bar_date"]).reset_index(drop=True)
daily["avg_vol_20d"]   = daily.groupby("ticker")["volume"].transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
daily["vol_ratio_1d"]  = daily["volume"] / daily["avg_vol_20d"]
daily["prev_close"]    = daily.groupby("ticker")["close"].shift(1)
daily["day_chg_pct"]   = (daily["close"] - daily["prev_close"]) / daily["prev_close"] * 100
daily["high_5d"]       = daily.groupby("ticker")["high"].transform(lambda x: x.shift(1).rolling(5).max())
daily["low_5d"]        = daily.groupby("ticker")["low"].transform(lambda x: x.shift(1).rolling(5).min())
daily["range_5d_pct"]  = (daily["high_5d"] - daily["low_5d"]) / daily["close"] * 100
daily["high_20d"]      = daily.groupby("ticker")["high"].transform(lambda x: x.shift(1).rolling(20).max())
daily["dist_20d_high"] = (daily["close"] - daily["high_20d"]) / daily["high_20d"] * 100
daily["bar_date"]      = pd.to_datetime(daily["bar_date"])
daily_idx = daily.set_index(["ticker","bar_date"])
print(f"Daily bars: {len(daily):,}  |  tickers: {daily['ticker'].nunique()}")

with sqlite3.connect(FINVIZ_DB) as conn:
    snaps = pd.read_sql("SELECT ticker, category, SUBSTR(timestamp,1,10) AS date FROM snapshots", conn)
daily_presence = snaps.groupby(["ticker","date","category"]).size().reset_index(name="n")
print(f"Finviz snapshots: {len(snaps):,}  |  fechas: {snaps['date'].nunique()}  |  categorias: {snaps['category'].nunique()}")

with sqlite3.connect(FINVIZ_DB) as conn:
    bursts = pd.read_sql("""
        SELECT ticker, SUBSTR(timestamp,1,10) AS burst_date,
               MAX(CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL)) AS burst_chg_pct
        FROM snapshots WHERE category='Top Gainers'
          AND CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) >= 15
        GROUP BY ticker, SUBSTR(timestamp,1,10)
    """, conn)
bursts["burst_date"] = pd.to_datetime(bursts["burst_date"])
print(f"Bursts >=15%: {len(bursts)}")
trading_dates = daily.groupby("ticker")["bar_date"].apply(sorted).to_dict()

## 3. Dataset T-1

In [ ]:
def get_t1(ticker, burst_date):
    dates = trading_dates.get(ticker, [])
    before = [d for d in dates if d < burst_date]
    return before[-1] if before else None

records = []
for _, row in bursts.iterrows():
    t, burst_date, burst_chg = row["ticker"], row["burst_date"], row["burst_chg_pct"]
    t1 = get_t1(t, burst_date)
    if t1 is None: continue
    try:
        d_row = daily_idx.loc[(t, t1)]
    except KeyError:
        continue
    t1_str = t1.strftime("%Y-%m-%d")
    pres = daily_presence[(daily_presence["ticker"]==t) & (daily_presence["date"]==t1_str)]
    cats_t1 = set(pres["category"].tolist())
    try:
        t_row = daily_idx.loc[(t, burst_date)]
        ret_open  = (t_row["open"]  - d_row["close"]) / d_row["close"] * 100
        ret_close = (t_row["close"] - d_row["close"]) / d_row["close"] * 100
    except KeyError:
        ret_open = ret_close = None
    records.append({
        "ticker": t, "burst_date": burst_date, "t1_date": t1, "burst_chg_pct": burst_chg,
        "t1_close":         d_row["close"],
        "t1_vol_ratio":     d_row.get("vol_ratio_1d"),
        "t1_day_chg_pct":   d_row.get("day_chg_pct"),
        "t1_range_5d_pct":  d_row.get("range_5d_pct"),
        "t1_dist_20d_high": d_row.get("dist_20d_high"),
        "ret_t_open":       round(ret_open,  2) if ret_open  is not None else None,
        "ret_t_close":      round(ret_close, 2) if ret_close is not None else None,
        "in_uv_t1":         "Unusual Volume" in cats_t1,
        "in_nh_t1":         "New High"        in cats_t1,
        "in_mv_t1":         "Most Volatile"   in cats_t1,
        "in_tg_t1":         "Top Gainers"     in cats_t1,
        "in_tl_t1":         "Top Losers"      in cats_t1,
        "in_any_t1":        len(cats_t1) > 0,
        "n_sig_cats_t1":    len(cats_t1 & set(SIGNAL_CATS)),
    })

t1_df = pd.DataFrame(records)
has = t1_df["ret_t_open"].notna()
print(f"Dataset T-1: {len(t1_df)} bursts  |  con retorno: {has.sum()}")
print(f"  En Finviz el dia T-1 (alguna cat):  {t1_df['in_any_t1'].sum()} ({t1_df['in_any_t1'].mean()*100:.0f}%)")
print(f"  En Unusual Volume T-1:              {t1_df['in_uv_t1'].sum()} ({t1_df['in_uv_t1'].mean()*100:.0f}%)")
print(f"  En New High T-1:                    {t1_df['in_nh_t1'].sum()} ({t1_df['in_nh_t1'].mean()*100:.0f}%)")
print(f"  En Top Gainers T-1 (multi-day):     {t1_df['in_tg_t1'].sum()} ({t1_df['in_tg_t1'].mean()*100:.0f}%)")

## 4. P&L entrada cierre T-1

In [ ]:
has = t1_df["ret_t_open"].notna()
df = t1_df[has].copy()
print("=" * 60)
print(f"P&L ENTRADA CIERRE T-1  (n={len(df)})")
print("=" * 60)
print(f"\nUNIVERSO COMPLETO:")
print(f"  T open  median={df['ret_t_open'].median():+.1f}%  mean={df['ret_t_open'].mean():+.1f}%  WR={( df['ret_t_open']>0).mean()*100:.0f}%")
print(f"  T close median={df['ret_t_close'].median():+.1f}%  mean={df['ret_t_close'].mean():+.1f}%  WR={( df['ret_t_close']>0).mean()*100:.0f}%")

print(f"\nPOR MAGNITUD DEL BURST:")
for lo, hi in [(15,30),(30,50),(50,100),(100,999)]:
    sub = df[(df["burst_chg_pct"]>=lo) & (df["burst_chg_pct"]<hi)]
    if len(sub) < 3: continue
    print(f"  Burst {lo}-{hi}% (n={len(sub)}):  open={sub['ret_t_open'].median():+.1f}%  close={sub['ret_t_close'].median():+.1f}%  WR_open={( sub['ret_t_open']>0).mean()*100:.0f}%")

print(f"\nPOR PRESENCIA EN FINVIZ T-1:")
for col, label in [("in_uv_t1","Unusual Volume"),("in_nh_t1","New High"),("in_tg_t1","Top Gainers T-1"),("in_any_t1","Cualquier cat")]:
    yes = df[df[col]==True]
    no  = df[df[col]==False]
    if len(yes) < 3: continue
    print(f"  {label} (n={len(yes)}):  open={yes['ret_t_open'].median():+.1f}% vs no:{no['ret_t_open'].median():+.1f}%  close={yes['ret_t_close'].median():+.1f}% vs no:{no['ret_t_close'].median():+.1f}%")

print(f"\nPOR COMPRESION RANGO T-1:")
hr = has & t1_df["t1_range_5d_pct"].notna()
low_c  = t1_df[hr & (t1_df["t1_range_5d_pct"] < 15)]
high_c = t1_df[hr & (t1_df["t1_range_5d_pct"] >= 15)]
if len(low_c)>=3:  print(f"  Rango <15% comprimido (n={len(low_c)}): open={low_c['ret_t_open'].median():+.1f}%  close={low_c['ret_t_close'].median():+.1f}%")
if len(high_c)>=3: print(f"  Rango >=15% (n={len(high_c)}):          open={high_c['ret_t_open'].median():+.1f}%  close={high_c['ret_t_close'].median():+.1f}%")

## 5. Grupo de control — tasa de burst condicional

In [ ]:
# Grupo control: todos los ticker-days Finviz -> burstaron al dia siguiente?
burst_lookup = {}
for _, row in bursts.iterrows():
    d = row["burst_date"].strftime("%Y-%m-%d")
    burst_lookup.setdefault(d, set()).add(row["ticker"])

all_dates = sorted(snaps["date"].unique())
ctrl_records = []
for i, d_str in enumerate(all_dates[:-1]):
    next_d = all_dates[i+1]
    today_tickers = set(snaps[snaps["date"]==d_str]["ticker"].unique())
    burst_tomorrow = burst_lookup.get(next_d, set())
    for t in today_tickers:
        cats = set(snaps[(snaps["ticker"]==t) & (snaps["date"]==d_str)]["category"].tolist())
        burst_next = t in burst_tomorrow
        try:
            d_row  = daily_idx.loc[(t, pd.Timestamp(d_str))]
            nd_row = daily_idx.loc[(t, pd.Timestamp(next_d))]
            r_open  = (nd_row["open"]  - d_row["close"]) / d_row["close"] * 100
            r_close = (nd_row["close"] - d_row["close"]) / d_row["close"] * 100
        except: r_open = r_close = None
        ctrl_records.append({
            "ticker": t, "date": d_str, "burst_next": burst_next,
            "in_uv":  "Unusual Volume" in cats,
            "in_nh":  "New High"        in cats,
            "in_tg":  "Top Gainers"     in cats,
            "in_mv":  "Most Volatile"   in cats,
            "in_ma":  "Most Active"     in cats,
            "n_sig":  len(cats & set(SIGNAL_CATS)),
            "ret_open": round(r_open,2) if r_open is not None else None,
            "ret_close":round(r_close,2) if r_close is not None else None,
        })

ctrl = pd.DataFrame(ctrl_records)
base_rate = ctrl["burst_next"].mean()
print(f"Grupo control: {len(ctrl)} ticker-days  |  tasa base burst manana: {base_rate*100:.1f}%")
print()
print(f"TASA DE BURST DIA SIGUIENTE SEGUN SENAL HOY:")
print(f"  {'Senal':<35} {'n':>6} {'burst_rate':>12} {'lift':>8}")
print("─"*65)
for col, label in [("in_uv","Unusual Volume"),("in_nh","New High"),("in_tg","Top Gainers"),("in_mv","Most Volatile"),("in_ma","Most Active")]:
    sub = ctrl[ctrl[col]==True]
    if len(sub) < 5: continue
    rate = sub["burst_next"].mean()
    print(f"  {label:<35} {len(sub):>6} {rate*100:>10.1f}% {rate/base_rate:>8.1f}x")
print()
print("COMBINACIONES:")
for combo, label in [
    (["in_uv","in_nh"],  "UV + New High"),
    (["in_uv","in_tg"],  "UV + Top Gainers"),
    (["in_nh","in_tg"],  "New High + Top Gainers"),
    (["in_uv","in_mv"],  "UV + Most Volatile"),
]:
    mask = pd.Series([True]*len(ctrl))
    for c in combo: mask = mask & (ctrl[c]==True)
    sub = ctrl[mask]
    if len(sub) < 3: continue
    rate = sub["burst_next"].mean()
    print(f"  {label:<35} {len(sub):>6} {rate*100:>10.1f}% {rate/base_rate:>8.1f}x")

## 6. Score T-1

In [ ]:
def t1_score(row, prefix="t1_"):
    s = 0
    uv  = row.get(f"in_uv_{prefix}",  row.get("in_uv",  False))
    nh  = row.get(f"in_nh_{prefix}",  row.get("in_nh",  False))
    tg  = row.get(f"in_tg_{prefix}",  row.get("in_tg",  False))
    mv  = row.get(f"in_mv_{prefix}",  row.get("in_mv",  False))
    if uv: s += 3
    if nh: s += 2
    if tg: s += 2
    if mv: s += 1
    r5 = row.get("t1_range_5d_pct") or row.get("range_5d_pct")
    if r5 is not None and not (isinstance(r5, float) and pd.isna(r5)):
        if r5 < 10: s += 2
        elif r5 < 15: s += 1
    vr = row.get("t1_vol_ratio") or row.get("vol_ratio_1d")
    if vr is not None and not (isinstance(vr, float) and pd.isna(vr)):
        if vr > 2: s += 2
        elif vr > 1: s += 1
    dh = row.get("t1_dist_20d_high") or row.get("dist_20d_high")
    if dh is not None and not (isinstance(dh, float) and pd.isna(dh)):
        if dh > -5: s += 2
        elif dh > -15: s += 1
    return s

t1_df["score"] = t1_df.apply(lambda r: t1_score(r, "t1_"), axis=1)
ctrl["score"]  = ctrl.apply(lambda r: t1_score(r, ""), axis=1)

has = t1_df["ret_t_open"].notna()
print("SCORE T-1 — DISTRIBUCION Y P&L")
print("="*65)
print(f"{'Score':>7} {'n_bursts':>9} {'n_ctrl':>8} {'rate_ctrl%':>11} {'lift':>7} {'T_open_med%':>13} {'WR_open%':>10}")
print("─"*65)
for s in sorted(t1_df["score"].unique()):
    nb_ = (t1_df["score"]==s).sum()
    nc_ = (ctrl["score"]==s).sum()
    nc_b = ctrl[(ctrl["score"]==s) & ctrl["burst_next"]].shape[0]
    rate = nc_b/nc_ if nc_>0 else 0
    lift = rate/base_rate if base_rate>0 else 0
    sub_b = t1_df[has & (t1_df["score"]==s)]
    med = sub_b["ret_t_open"].median() if len(sub_b)>=3 else float("nan")
    wr  = (sub_b["ret_t_open"]>0).mean()*100 if len(sub_b)>=3 else float("nan")
    print(f"  {s:>5}  {nb_:>9}  {nc_:>8}  {rate*100:>9.1f}%  {lift:>6.1f}x  {med:>12.1f}%  {wr:>9.0f}%")

## 7. Visualización

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Analisis T-1 Pre-Burst | Finviz + Barra Diaria", fontsize=13)
has = t1_df["ret_t_open"].notna()

ax = axes[0,0]
yes = t1_df[has & (t1_df["in_uv_t1"]==True)]["ret_t_open"].clip(-30,200)
no_ = t1_df[has & (t1_df["in_uv_t1"]==False)]["ret_t_open"].clip(-30,200)
bins = np.linspace(-30, 150, 30)
ax.hist(no_,  bins=bins, alpha=0.6, color="gray",  label=f"No UV T-1 (n={len(no_)})", density=True)
ax.hist(yes,  bins=bins, alpha=0.7, color="green", label=f"UV T-1 (n={len(yes)})",   density=True)
ax.axvline(0, color="red", linewidth=1)
ax.set_xlabel("Return T open (%)"); ax.set_title("UV T-1 -> Return apertura T"); ax.legend(fontsize=8)

ax = axes[0,1]
ax.scatter(t1_df.loc[has,"burst_chg_pct"].clip(upper=300), t1_df.loc[has,"ret_t_open"], alpha=0.4, s=15, color="steelblue")
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Burst % dia T"); ax.set_ylabel("Return T-1 close -> T open (%)"); ax.set_title("Magnitud burst vs ganancia")

ax = axes[0,2]
ctrl_rates = ctrl.groupby("score")["burst_next"].mean()*100
ctrl_counts = ctrl.groupby("score").size()
ax.bar(ctrl_rates.index, ctrl_rates.values, color="steelblue", alpha=0.8)
ax.axhline(base_rate*100, color="red", linestyle="--", label=f"Base rate {base_rate*100:.1f}%")
ax.set_xlabel("Score T-1"); ax.set_ylabel("Tasa burst dia siguiente (%)"); ax.set_title("Score -> prob burst")
ax.legend(fontsize=8)

ax = axes[1,0]
hr = has & t1_df["t1_range_5d_pct"].notna()
ax.scatter(t1_df.loc[hr,"t1_range_5d_pct"].clip(upper=80), t1_df.loc[hr,"ret_t_open"], alpha=0.5, s=15, color="orange")
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.axvline(15, color="gray", linewidth=1, linestyle="--", label="15% threshold")
ax.set_xlabel("Range 5d T-1 (%)"); ax.set_ylabel("Return T open (%)"); ax.set_title("Compresion T-1 vs ganancia"); ax.legend(fontsize=8)

ax = axes[1,1]
hv = has & t1_df["t1_vol_ratio"].notna()
ax.scatter(t1_df.loc[hv,"t1_vol_ratio"].clip(upper=10), t1_df.loc[hv,"ret_t_open"], alpha=0.5, s=15, color="purple")
ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Vol ratio T-1"); ax.set_ylabel("Return T open (%)"); ax.set_title("Vol spike T-1 vs ganancia")

ax = axes[1,2]
scores = sorted(t1_df.loc[has,"score"].unique())
groups = [t1_df.loc[has & (t1_df["score"]==s),"ret_t_open"].clip(-50,200).values for s in scores if (has & (t1_df["score"]==s)).sum()>=3]
labels = [str(s) for s in scores if (has & (t1_df["score"]==s)).sum()>=3]
if groups:
    bp = ax.boxplot(groups, labels=labels, patch_artist=True)
    for p in bp["boxes"]: p.set_facecolor("steelblue"); p.set_alpha(0.6)
    ax.axhline(0, color="red", linewidth=1, linestyle="--")
ax.set_xlabel("Score T-1"); ax.set_ylabel("Return T open (%)"); ax.set_title("Retorno por score T-1")

plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
fig.savefig("figures/t1_preburst.png", dpi=130, bbox_inches="tight")
plt.show()
print("Guardado en figures/t1_preburst.png")

## 8. Conclusiones operativas

In [ ]:
print("=" * 65)
print("CONCLUSIONES OPERATIVAS — T-1 PREBURST")
print("=" * 65)
has = t1_df["ret_t_open"].notna()
df_ = t1_df[has].copy()
n = len(df_)

print(f"\nDataset: {n} bursts con T-1 data")
print(f"Tasa base burst dia siguiente (universo Finviz): {base_rate*100:.1f}%")
print(f"Historico disponible: {snaps['date'].nunique()} dias (desde 2026-03-24)")
print()

for thresh in [3, 4, 5, 6]:
    hs = df_[df_["score"] >= thresh]
    if len(hs) < 3: continue
    ctrl_hs = ctrl[ctrl["score"] >= thresh]
    ctrl_rate = ctrl_hs["burst_next"].mean()*100 if len(ctrl_hs)>0 else 0
    print(f"SCORE >= {thresh} (n_burst={len(hs)}, n_ctrl={len(ctrl_hs)}):")
    print(f"  Tasa burst en ctrl:     {ctrl_rate:.1f}%  (lift {ctrl_rate/base_rate/100:.1f}x)")
    print(f"  Return T open  median:  {hs['ret_t_open'].median():+.1f}%")
    print(f"  Return T close median:  {hs['ret_t_close'].median():+.1f}%")
    print(f"  WR T open > 0:          {(hs['ret_t_open']>0).mean()*100:.0f}%")
    losses = (hs["ret_t_open"] < -10).sum()
    print(f"  Perdidas > -10%:        {losses} ({losses/len(hs)*100:.0f}%)")
    print()

print(f"LIMITACION PRINCIPAL:")
print(f"  Solo {snaps['date'].nunique()} dias de historico — insuficiente para validacion OOS")
print(f"  {100-t1_df['in_any_t1'].mean()*100:.0f}% de los bursts NO estaban en Finviz el dia anterior")
print(f"  Finviz cubre ~{snaps['ticker'].nunique()} tickers — el universo small-cap es 3000+")
print(f"  Recomendacion: acumular 3+ meses de datos antes de implementar en produccion")